In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "DIQ_E.xpt"
csv_file = DATA_DIR / "DIQ_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 DIQ_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "DIQ010",
    "DID040",
    "DIQ220",
    "DIQ160",
    "DIQ170",
    "DIQ180",
    "DIQ190A",
    "DIQ190B",
    "DIQ190C",
    "DIQ200A",
    "DIQ200B",
    "DIQ200C",
    "DIQ050",
    "DID060",
    "DIQ060U",
    "DID070",
    "DIQ230",
    "DIQ240",
    "DID250",
    "DID260",
    "DIQ260U",
    "DID270",
    "DIQ280",
    "DIQ290",
    "DIQ300S",
    "DIQ300D",
    "DID310S",
    "DID310D",
    "DID320",
    "DID330",
    "DID340",
    "DID350",
    "DIQ350U",
    "DIQ360",
    "DIQ080"
]


if df.shape != (9666, 36):
    raise ValueError(
        f"Unexpected DIQ_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "DIQ_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: DIQ_E structure = "
    "9,666 rows x 36 columns."
)


# ============================================================
# 4. CDC CHECK ITEMS - NOT RELEASED COLUMNS
# ============================================================

#
# CDC HTML contains several questionnaire routing/check items.
# They are NOT released variables in DIQ_E.xpt.
#
# Do NOT add them to the CSV.
#

CHECK_ITEMS = [
    "DIQ219",
    "DIQ159",
    "DID065",
    "DID229",
    "DIQ295"
]


for col in CHECK_ITEMS:

    if col in df.columns:

        raise ValueError(
            f"Unexpected CHECK ITEM found in XPT: {col}"
        )


print(
    "PASS: CDC routing/check items correctly "
    "absent from released XPT."
)


# ============================================================
# 5. FIELD CLASSIFICATION
# ============================================================

#
# DIQ280 is the ONLY genuine decimal field in the
# attached DIQ_E XPT.
#
# It is self-reported A1C.
#
# Examples:
#
# 6.5 -> 6.5
# 7.0 -> 7.0
# 999.0 -> 999.0
#
# DO NOT convert DIQ280 to Int64.
#
#
# All other released variables contain whole-number
# identifiers, questionnaire codes, counts, ages,
# or whole-number entered values.
#

DECIMAL_COLUMNS = [
    "DIQ280"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in DECIMAL_COLUMNS
]


print(
    "\nInteger-style fields:",
    len(INTEGER_COLUMNS)
)

print(
    "Decimal fields preserved AS IS:",
    DECIMAL_COLUMNS
)


# ============================================================
# 6. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny artifacts:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 7. EXPECTED TINY PATTERN
# ============================================================

EXPECTED_TINY_COUNTS = {
    "DID250": 11,
    "DID260": 125,
    "DID270": 62,
    "DID340": 226,
    "DID350": 125
}


EXPECTED_TINY_TOTAL = 549


actual_tiny_counts = {
    col: int(count)
    for col, count in tiny_counts.items()
    if count > 0
}


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny artifact pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-artifact count."
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected true numeric zeros were already "
        "decoded in the original XPT."
    )


print(
    "\nPASS: Exact tiny artifact pattern confirmed."
)


# ============================================================
# 8. CDC ZERO-SAFETY CHECK
# ============================================================

#
# CDC documentation:
#
# DID250:
# 0 = None
#
# DID260:
# 0 = Never
#
# DID270:
# 0 = Not tested in last 12 months
#
# DID340:
# 0 = None
#
# DID350:
# 0 = None
#
# Therefore ONLY these exact tiny values are approved
# for tiny -> zero restoration.
#

APPROVED_ZERO_COLUMNS = {
    "DID250",
    "DID260",
    "DID270",
    "DID340",
    "DID350"
}


unexpected_tiny_columns = (
    set(actual_tiny_counts)
    -
    APPROVED_ZERO_COLUMNS
)


if unexpected_tiny_columns:

    raise ValueError(
        "STOP: Tiny values found in unexpected columns:\n"
        + str(
            sorted(unexpected_tiny_columns)
        )
    )


print(
    "PASS: Tiny artifacts occur only in "
    "CDC zero-permitted variables."
)


# ============================================================
# 9. RESTORE VALIDATED ZEROS
# ============================================================

print(
    "\n--- ZERO RESTORATION ---"
)


for col, expected_count in (
    EXPECTED_TINY_COUNTS.items()
):

    mask = df[col].eq(
        TINY_VALUE
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{actual_count:,} tiny -> 0"
    )


print(
    "PASS: All 549 validated zeros restored."
)


# ============================================================
# 10. VERIFY FINAL ZERO COUNTS
# ============================================================

EXPECTED_ZERO_COUNTS = {
    "DID250": 11,
    "DID260": 125,
    "DID270": 62,
    "DID340": 226,
    "DID350": 125
}


for col, expected_count in (
    EXPECTED_ZERO_COUNTS.items()
):

    actual_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: zero count differs.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


print(
    "PASS: Restored zero counts are exact."
)


# ============================================================
# 11. VERIFY NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "Remaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifact remains."
    )


print(
    "PASS: No tiny artifacts remain."
)


# ============================================================
# 12. SEQN VALIDATION
# ============================================================

seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "\nPASS: SEQN range = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 13. FRACTIONAL / DECIMAL AUDIT
# ============================================================

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:

        fractional_columns[
            col
        ] = fractional_count


EXPECTED_FRACTIONAL_COUNTS = {
    "DIQ280": 26
}


print(
    "\n--- GENUINE FRACTIONAL VALUE CHECK ---"
)

print(
    fractional_columns
)


if fractional_columns != EXPECTED_FRACTIONAL_COUNTS:

    raise ValueError(
        "Unexpected decimal pattern.\n"
        f"Expected: {EXPECTED_FRACTIONAL_COUNTS}\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "PASS: DIQ280 is the only genuine decimal field."
)

print(
    "PASS: DIQ280 genuine fractional responses = 26."
)


# ============================================================
# 14. CDC CATEGORICAL FREQUENCY CHECKS
# ============================================================

#
# These are exact CDC/XPT frequencies for variables
# with small categorical code sets.
#

EXPECTED_CATEGORICAL = {

    "DIQ010": (
        {1: 777, 2: 8774, 3: 106, 9: 9},
        0
    ),

    "DIQ220": (
        {1: 15, 2: 22, 3: 12, 4: 18, 5: 75},
        9524
    ),

    "DIQ160": (
        {1: 242, 2: 6052, 9: 11},
        3361
    ),

    "DIQ170": (
        {1: 735, 2: 5655, 9: 14},
        3262
    ),

    "DIQ180": (
        {1: 2420, 2: 3735, 9: 249},
        3262
    ),

    "DIQ190A": (
        {1: 1734, 2: 5436, 9: 3},
        2493
    ),

    "DIQ190B": (
        {1: 2169, 2: 5000, 9: 4},
        2493
    ),

    "DIQ190C": (
        {1: 1967, 2: 5203, 9: 3},
        2493
    ),

    "DIQ200A": (
        {1: 3427, 2: 3746},
        2493
    ),

    "DIQ200B": (
        {1: 3339, 2: 3833, 9: 1},
        2493
    ),

    "DIQ200C": (
        {1: 3399, 2: 3773, 9: 1},
        2493
    ),

    "DIQ050": (
        {1: 219, 2: 9447},
        0
    ),

    "DIQ060U": (
        {1: 30, 2: 182},
        9454
    ),

    "DID070": (
        {1: 616, 2: 505, 9: 4},
        8541
    ),

    "DIQ230": (
        {1: 280, 2: 63, 3: 75, 4: 82, 5: 275, 9: 2},
        8889
    ),

    "DIQ240": (
        {1: 589, 2: 188},
        8889
    ),

    "DIQ260U": (
        {1: 423, 2: 151, 3: 47, 4: 25},
        9020
    ),

    "DIQ290": (
        {
            1: 128,
            2: 92,
            3: 5,
            4: 5,
            5: 8,
            6: 114,
            77: 1,
            99: 160
        },
        9153
    ),

    "DIQ350U": (
        {1: 390, 2: 140, 3: 43, 4: 10},
        9083
    ),

    "DIQ360": (
        {1: 90, 2: 393, 3: 105, 4: 114, 5: 62, 9: 5},
        8897
    ),

    "DIQ080": (
        {1: 173, 2: 582, 9: 14},
        8897
    )
}


print(
    "\n--- CDC CATEGORICAL CHECKS ---"
)


for col, (
    expected_counts,
    expected_missing
) in EXPECTED_CATEGORICAL.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v in actual_counts.items()
    }


    if actual_counts != expected_counts:

        raise ValueError(
            f"{col} frequency mismatch.\n"
            f"Expected: {expected_counts}\n"
            f"Actual:   {actual_counts}"
        )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{col} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 15. CDC RANGE / SPECIAL-CODE VALIDATION
# ============================================================

def check_range_variable(
    col,
    low,
    high,
    expected_range_count,
    special_counts,
    expected_missing
):

    s = df[col]


    range_count = int(
        s.between(
            low,
            high
        )
        .sum()
    )


    if range_count != expected_range_count:

        raise ValueError(
            f"{col}: range count differs.\n"
            f"Expected {low}-{high}: "
            f"{expected_range_count:,}\n"
            f"Actual: {range_count:,}"
        )


    for code, expected_count in (
        special_counts.items()
    ):

        actual_count = int(
            s.eq(code)
            .sum()
        )


        if actual_count != expected_count:

            raise ValueError(
                f"{col}: code {code} count differs.\n"
                f"Expected: {expected_count:,}\n"
                f"Actual:   {actual_count:,}"
            )


    actual_missing = int(
        s.isna()
        .sum()
    )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{col}: missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {col}"
    )


print(
    "\n--- CDC RANGE/SPECIAL-CODE CHECKS ---"
)


check_range_variable(
    "DID040",
    1,
    79,
    754,
    {
        80: 12,
        666: 4,
        777: 0,
        999: 7
    },
    8889
)


check_range_variable(
    "DID060",
    1,
    74,
    212,
    {
        666: 5,
        777: 0,
        999: 2
    },
    9447
)


check_range_variable(
    "DID250",
    1,
    52,
    575,
    {
        0: 11,
        7777: 0,
        9999: 3
    },
    9077
)


check_range_variable(
    "DID260",
    1,
    31,
    646,
    {
        0: 125,
        666: 5,
        777: 0,
        999: 1
    },
    8889
)


check_range_variable(
    "DID270",
    1,
    24,
    363,
    {
        0: 62,
        666: 202,
        777: 1,
        999: 149
    },
    8889
)


#
# DIQ280 = TRUE DECIMAL A1C FIELD
#

check_range_variable(
    "DIQ280",
    1.0,
    18.0,
    214,
    {
        777: 0,
        999: 299
    },
    9153
)


check_range_variable(
    "DIQ300S",
    70,
    199,
    443,
    {
        7777: 2,
        9999: 324
    },
    8897
)


check_range_variable(
    "DIQ300D",
    20,
    140,
    431,
    {
        7777: 1,
        9999: 337
    },
    8897
)


check_range_variable(
    "DID310S",
    70,
    180,
    214,
    {
        6666: 414,
        7777: 0,
        9999: 141
    },
    8897
)


check_range_variable(
    "DID310D",
    10,
    140,
    210,
    {
        6666: 414,
        7777: 0,
        9999: 145
    },
    8897
)


check_range_variable(
    "DID320",
    32,
    350,
    83,
    {
        5555: 140,
        6666: 22,
        7777: 0,
        9999: 524
    },
    8897
)


check_range_variable(
    "DID330",
    40,
    200,
    57,
    {
        6666: 286,
        7777: 0,
        9999: 264
    },
    9059
)


check_range_variable(
    "DID340",
    1,
    144,
    481,
    {
        0: 226,
        7777: 0,
        9999: 3
    },
    8956
)


check_range_variable(
    "DID350",
    1,
    52,
    583,
    {
        0: 125,
        7777: 0,
        9999: 2
    },
    8956
)


# ============================================================
# 16. DIQ280 STRICT DECIMAL CHECK
# ============================================================

#
# Actual attached XPT:
#
# Valid nonmissing       = 513
# Numeric A1C 1.0-18.0   = 214
# 999 Don't know         = 299
# Missing                = 9,153
#
# Genuine fractional responses = 26
#
# Whole-number float-style values = 487
#
# IMPORTANT:
#
# 7.0 remains "7.0"
# 999.0 remains "999.0"
#
# because DIQ280 is deliberately preserved as
# a float/decimal field.
#

diq280 = df["DIQ280"]


diq280_nonmissing = int(
    diq280.notna()
    .sum()
)


if diq280_nonmissing != 513:

    raise ValueError(
        "DIQ280 valid count differs."
    )


diq280_values = (
    diq280
    .dropna()
    .to_numpy()
)


diq280_fractional = int(
    (
        ~np.isclose(
            diq280_values,
            np.round(diq280_values),
            rtol=0,
            atol=1e-12
        )
    ).sum()
)


if diq280_fractional != 26:

    raise ValueError(
        "DIQ280 fractional-value count differs."
    )


diq280_source_dot_zero = sum(
    str(
        float(value)
    ).endswith(".0")
    for value in diq280.dropna()
)


if diq280_source_dot_zero != 487:

    raise ValueError(
        "DIQ280 source .0 count differs."
    )


print(
    "\nPASS: DIQ280 genuine decimal field verified."
)

print(
    "      Nonmissing = 513"
)

print(
    "      Genuine fractional values = 26"
)

print(
    "      Legitimate source '.0' values = 487"
)


# ============================================================
# 17. BUILD EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# expected_df =
#
# ORIGINAL XPT
#
# +
# ONLY these five validated tiny -> zero corrections:
#
# DID250
# DID260
# DID270
# DID340
# DID350
#
# NOTHING ELSE.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 18. CLEAN INTEGER-STYLE FIELDS
# ============================================================

#
# Convert every proven whole-number questionnaire
# field to Int64.
#
# DIQ280 is deliberately EXCLUDED.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) > 0:

        if not np.all(
            np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=0
            )
        ):

            raise ValueError(
                f"STOP: {col} contains a "
                "non-whole value."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Integer/code/count fields cleaned."
)

print(
    "PASS: DIQ280 left untouched as float."
)


# ============================================================
# 19. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col in INTEGER_COLUMNS:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )

    else:

        current_values = (
            df[col]
            .to_numpy()
        )


    equal = np.isclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not bool(
        np.all(equal)
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected pre-export changes:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export values equal expected XPT "
    "after ONLY the 549 zero restorations."
)


# ============================================================
# 20. EXPORT CSV
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# NO:
#
# round()
# float_format
# decimal quantization
# recalculation
#
# Missing values -> blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 21. RE-READ CSV WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 22. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure matches XPT."
)


# ============================================================
# 23. EXACT NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 36 columns numerically match "
    "expected XPT values exactly."
)


# ============================================================
# 24. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += (
        differences
    )


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing-position fidelity failure:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 25. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    raw_rows = list(
        csv.DictReader(f)
    )


if len(raw_rows) != 9666:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 26. STRICT RAW CSV TOKEN FIDELITY
# ============================================================

#
# Integer/code/count variables:
#
# expected = str(int(source_value))
#
#
# DIQ280:
#
# expected = str(float(source_value))
#
#
# Examples:
#
# SEQN:
# 41475.0 -> "41475"
#
# DID250:
# tiny -> "0"
#
# DIQ280:
# 6.5   -> "6.5"
# 7.0   -> "7.0"
# 999.0 -> "999.0"
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        if pd.isna(
            source_value
        ):

            expected_text = ""

        elif col in DECIMAL_COLUMNS:

            expected_text = str(
                float(source_value)
            )

        else:

            expected_text = str(
                int(source_value)
            )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:
        print(example)


    raise ValueError(
        "DIQ_E raw CSV formatting differs."
    )


print(
    "PASS: Every raw CSV token matches "
    "the expected XPT-derived representation."
)


# ============================================================
# 27. UNWANTED .0 IN INTEGER FIELDS
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in INTEGER_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nUnwanted '.0' in integer fields:",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        dot_zero_examples
    )

    raise ValueError(
        "Unwanted .0 remains in integer/code fields."
    )


print(
    "PASS: No unwanted .0 in integer fields."
)


# ============================================================
# 28. DIQ280 LEGITIMATE .0 CHECK
# ============================================================

diq280_csv_dot_zero = sum(
    row["DIQ280"].endswith(".0")
    for row in raw_rows
    if row["DIQ280"] != ""
)


print(
    "DIQ280 legitimate '.0' values:",
    diq280_csv_dot_zero
)


if diq280_csv_dot_zero != 487:

    raise ValueError(
        "DIQ280 legitimate .0 formatting changed.\n"
        "Expected: 487\n"
        f"Actual:   {diq280_csv_dot_zero}"
    )


print(
    "PASS: DIQ280 legitimate .0 values preserved."
)


# ============================================================
# 29. STRICT DIQ280 RAW DECIMAL CHECK
# ============================================================

diq280_raw_mismatches = 0
diq280_examples = []


for i, row in enumerate(
    raw_rows
):

    source_value = (
        expected_df.iloc[i]["DIQ280"]
    )


    expected_text = (
        ""
        if pd.isna(source_value)
        else str(
            float(source_value)
        )
    )


    actual_text = (
        row["DIQ280"]
    )


    if actual_text != expected_text:

        diq280_raw_mismatches += 1


        if len(
            diq280_examples
        ) < 20:

            diq280_examples.append(
                {
                    "csv_row": i + 2,
                    "expected": expected_text,
                    "actual": actual_text
                }
            )


print(
    "DIQ280 raw decimal differences:",
    diq280_raw_mismatches
)


if diq280_raw_mismatches != 0:

    print(
        diq280_examples
    )

    raise ValueError(
        "DIQ280 decimal representation changed."
    )


print(
    "PASS: Every DIQ280 value preserved AS IS."
)


# ============================================================
# 30. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. TINY ARTIFACT STRING CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact remains."
)


# ============================================================
# 32. FINAL ZERO RECHECK
# ============================================================

for col, expected_count in (
    EXPECTED_ZERO_COUNTS.items()
):

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"Final {col} zero count changed."
        )


print(
    "PASS: Final restored-zero counts exact."
)


# ============================================================
# 33. FINAL DIQ280 RECHECK
# ============================================================

final_diq280 = csv_df["DIQ280"]


if int(
    final_diq280.notna()
    .sum()
) != 513:

    raise ValueError(
        "Final DIQ280 valid count changed."
    )


if int(
    final_diq280
    .between(
        1.0,
        18.0
    )
    .sum()
) != 214:

    raise ValueError(
        "Final DIQ280 1.0-18.0 count changed."
    )


if int(
    final_diq280
    .eq(999)
    .sum()
) != 299:

    raise ValueError(
        "Final DIQ280=999 count changed."
    )


if int(
    final_diq280.isna()
    .sum()
) != 9153:

    raise ValueError(
        "Final DIQ280 missing count changed."
    )


print(
    "PASS: Final DIQ280 distribution unchanged."
)


# ============================================================
# 34. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("DIQ_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nValidated tiny -> zero:"
)

print(
    "DID250 = 11"
)

print(
    "DID260 = 125"
)

print(
    "DID270 = 62"
)

print(
    "DID340 = 226"
)

print(
    "DID350 = 125"
)

print(
    "TOTAL  = 549"
)


print(
    "\nGenuine decimal variables:"
)

print(
    "DIQ280"
)

print(
    "DIQ280 genuine fractional values = 26"
)

print(
    "DIQ280 legitimate '.0' values = 487"
)


print(
    "\nCDC CHECK ITEMS not added:"
)

for item in CHECK_ITEMS:
    print(item)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "DIQ280 raw differences:",
    diq280_raw_mismatches
)

print(
    "Unwanted .0 in integer fields:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "DIQ_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 DIQ_E XPT -> CSV
Rows: 9,666
Columns: 36
PASS: DIQ_E structure = 9,666 rows x 36 columns.
PASS: CDC routing/check items correctly absent from released XPT.

Integer-style fields: 35
Decimal fields preserved AS IS: ['DIQ280']

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifacts: 549

Tiny values by variable:
DID340    226
DID350    125
DID260    125
DID270     62
DID250     11

PASS: Exact tiny artifact pattern confirmed.
PASS: Tiny artifacts occur only in CDC zero-permitted variables.

--- ZERO RESTORATION ---
DID250    : 11 tiny -> 0
DID260    : 125 tiny -> 0
DID270    : 62 tiny -> 0
DID340    : 226 tiny -> 0
DID350    : 125 tiny -> 0
PASS: All 549 validated zeros restored.
PASS: Restored zero counts are exact.
Remaining tiny artifacts: 0
PASS: No tiny artifacts remain.

PASS: SEQN range = 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

--- GENUINE FRACTIONAL VALUE CHECK ---
{'DIQ280': 26}
PASS: DIQ280 is the only genuine decimal field